In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from typing import Optional
import torch.nn.functional as F
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import LRScheduler
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from dataset import Multi30kDataset
from lr_scheduler import NoamScheduler
from model import Transformer, make_src_mask, make_tgt_mask

from tqdm import tqdm

In [2]:
class LabelSmoothingLoss(nn.Module):
    def __init__(self, vocab_size: int, pad_idx: int = 1, smoothing: float = 0.1):
        super().__init__()

        self.vocab_size = vocab_size
        self.pad_idx = pad_idx
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:

        # logits: (B*T, V)
        # target: (B*T)

        log_probs = F.log_softmax(logits, dim=-1)   # (B*T, V)

        with torch.no_grad():
            # Create smoothed distribution
            true_dist = torch.zeros_like(log_probs)

            # Fill with smoothing value
            true_dist.fill_(self.smoothing / (self.vocab_size - 1))

            # Put confidence at correct indices
            true_dist.scatter_(1, target.unsqueeze(1), self.confidence)

            # Zero out pad positions
            true_dist[target == self.pad_idx] = 0

        # Compute loss
        loss = -(true_dist * log_probs).sum(dim=1)   # (B*T)

        # Ignore pad tokens in loss
        non_pad_mask = target != self.pad_idx
        loss = loss[non_pad_mask].mean()

        return loss

In [3]:
label_smoothing = LabelSmoothingLoss(1024, 1, 0.1)

logits = torch.randn(10, 128, 1024)   
target = torch.randint(0, 128, (10, 128))

logits = logits.view(-1, 1024)
target = target.view(-1)

loss = label_smoothing(logits, target)
print(loss)

tensor(7.4506)


In [20]:
logits.shape

torch.Size([1280, 1024])

In [4]:
language_dataset = Multi30kDataset(split = "train")
processed_dataset = language_dataset.process_data()

Building vocab, please wait...


100%|██████████| 29000/29000 [03:30<00:00, 137.94it/s]


In [5]:
class TranslationDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src, tgt = self.data[idx]
        return torch.tensor(src), torch.tensor(tgt)
    
def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, padding_value=1, batch_first=True)  # pad_idx = 1
    tgt_batch = pad_sequence(tgt_batch, padding_value=1, batch_first=True)
    return src_batch, tgt_batch

dataset_obj = TranslationDataset(processed_dataset)
dataloader = DataLoader(
    dataset_obj,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

In [6]:
def run_epoch(
    data_iter,
    model: Transformer,
    loss_fn: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler=None,
    epoch_num: int = 0,
    is_train: bool = True,
    device: str = "cpu",
) -> float:

    model.train() if is_train else model.eval()
    losses = []
    for _ in range(epoch_num):
        total_loss = 0

        for src, tgt in tqdm(data_iter):

            src = src.to(device)
            tgt = tgt.to(device)

            # masks
            src_mask = make_src_mask(src)
            tgt_mask = make_tgt_mask(tgt)

            # shift for teacher forcing
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]

            tgt_mask = make_tgt_mask(tgt_input)

            # forward
            logits = model(src, tgt_input, src_mask, tgt_mask)

            # reshape
            logits = logits.contiguous().view(-1, logits.shape[-1])
            tgt_output = tgt_output.contiguous().view(-1)

            # loss
            loss = loss_fn(logits, tgt_output)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                if scheduler is not None:
                    scheduler.step()

            total_loss += loss.item()
        losses.append(total_loss / len(data_iter))

    return sum(losses)/len(losses)

In [ ]:
src_vocab = len(language_dataset.de_vocab)
tgt_vocab = len(language_dataset.en_vocab)

transformer = Transformer(src_vocab_size = src_vocab, tgt_vocab_size = tgt_vocab,
                          d_model = 512, N = 6, num_heads = 8, d_ff = 2048,
                          dropout = 0.1)

label_smoothing = LabelSmoothingLoss(vocab_size = src_vocab, pad_idx = 1, smoothing = 0.1)

optimizer   = optim.Adam(transformer.parameters(), lr=1.0)
scheduler   = NoamScheduler(optimizer, d_model=512, warmup_steps=4000)

loss = run_epoch(data_iter = dataloader, model = transformer, loss_fn = label_smoothing, 
                 optimizer = optimizer, scheduler = scheduler, is_train = True, epoch_num = 1)

In [59]:
for idx in next(iter(dataloader))[0][0]:
    print(language_dataset.en_itos[idx.item()], end = " ")

<sos> a wooden near playhouse man bundt cleaning a window whilst wood street communicating men <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> 

In [ ]:
language_dataset.en_itos[idx]

'<unk>'

In [78]:
src_vocab = len(language_dataset.de_vocab)
tgt_vocab = len(language_dataset.en_vocab)

src_vocab, tgt_vocab

(18669, 9797)

In [20]:
src, tgt = next(iter(dataloader))

In [21]:
sample_src = src[0].unsqueeze(0)
target_src = tgt[0].unsqueeze(0)

In [23]:
target_src.shape

torch.Size([1, 20])

In [26]:
memory = transformer.encode(sample_src, make_src_mask(sample_src))

In [40]:
src_mask.shape

torch.Size([1, 1, 1, 18])

In [91]:
def greedy_decode(model, src, src_mask, max_len, start_symbol, end_symbol, device = "cpu"):
    src = src.to(device)
    src_mask = src_mask.to(device)
    
    # encode
    ys = torch.ones(src.size(0), 1).fill_(start_symbol).long().to(device)
    memory = model.encode(src, src_mask)

    # loop
    for _ in range(max_len):
        # decoding
        tgt_mask = make_tgt_mask(ys).to(device)
        out = model.decode(memory, src_mask, ys, tgt_mask)

        prob = out[:, -1, :] # (B, vocab)
        next_word = prob.argmax(dim=-1)

        ys = torch.cat([ys, next_word.unsqueeze(1)], dim=1)

        if (next_word == end_symbol).all():
            break
    return ys


sos_idx = 2
eos_idx = 3
device = "cpu"

src, tgt = next(iter(dataloader))
sample_src = src[0].unsqueeze(0)
src_mask = make_src_mask(sample_src).to(device)

src_vocab = len(language_dataset.de_vocab)
tgt_vocab = len(language_dataset.en_vocab)

transformer = Transformer(src_vocab_size = src_vocab, tgt_vocab_size = tgt_vocab,
                          d_model = 512, N = 6, num_heads = 8, d_ff = 2048,
                          dropout = 0.1)

prediction_idx = greedy_decode(transformer, sample_src, src_mask, max_len = 20, start_symbol = sos_idx, end_symbol = eos_idx, device = "cpu")
prediction_idx

tensor([[   2, 4309, 1816, 5205, 1816, 9260, 9260, 2800, 9260, 9260, 9260, 9260,
         5205, 9260, 9260, 9260, 1816, 9260, 5205, 9260, 1816]])

In [92]:
for i in prediction_idx.squeeze(0):
    print(language_dataset.en_itos[i.item()], end = " ")

<sos> posed pick spare pick waffles waffles knitting waffles waffles waffles waffles spare waffles waffles waffles pick waffles spare waffles pick 

In [87]:
prediction_idx.shape

torch.Size([1, 21])

In [63]:
ys.shape

torch.Size([1, 2])

In [ ]:
def greedy_decode(
    model: Transformer,
    src: torch.Tensor,
    src_mask: torch.Tensor,
    max_len: int,
    start_symbol: int,
    end_symbol: int,
    device: str = "cpu",
) -> torch.Tensor:
    """
    Generate a translation token-by-token using greedy decoding.

    Args:
        model        : Trained Transformer.
        src          : Source token indices, shape [1, src_len].
        src_mask     : shape [1, 1, 1, src_len].
        max_len      : Maximum number of tokens to generate.
        start_symbol : Vocabulary index of <sos>.
        end_symbol   : Vocabulary index of <eos>.
        device       : 'cpu' or 'cuda'.

    Returns:
        ys : Generated token indices, shape [1, out_len].
             Includes start_symbol; stops at (and includes) end_symbol
             or when max_len is reached.

    """
    # TODO: Task 3.3 — implement token-by-token greedy decoding
    raise NotImplementedError

In [81]:
src, tgt = next(iter(dataloader))

In [82]:
tgt.shape

torch.Size([32, 22])

In [83]:
src_mask = make_src_mask(src)
tgt_mask = make_tgt_mask(tgt)
src_mask.shape, tgt_mask.shape

(torch.Size([32, 1, 1, 27]), torch.Size([32, 1, 22, 22]))

In [86]:
logits = transformer(src, tgt, src_mask, tgt_mask)
logits.shape

torch.Size([32, 22, 9797])

In [95]:
tgt.view(-1).shape

torch.Size([704])

In [96]:
label_smoothing = LabelSmoothingLoss(vocab_size = src_vocab, pad_idx = 1, smoothing = 0.1)
label_smoothing(logits.view(-1, logits.shape[-1]), tgt.view(-1))

tensor(8.7842, grad_fn=<MeanBackward0>)